## Re-run `rMATS` stats for correct p-values

## Purpose: 

`ENCODE` provided `differential splicing` results thorugh their portal using `rMATS v3.2.1 beta`.

That version of `rMATS` has a bug that renders the p-values they have provided incorrect. 

Hence, we are re-running the `rMATS` stats model using the provided `rMATS` output files to replace the p-value column. 

## Packages and Options

In [1]:
import pandas as pd
pd.set_option('display.max_rows', 1000000)
pd.set_option('display.max_columns', 1000000)
pd.set_option('display.max_colwidth', 10000000)

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import glob, os, scipy.stats

## Load Data

In [2]:
columns_to_keep=['ID', 'IJC_SAMPLE_1', 'SJC_SAMPLE_1', 'IJC_SAMPLE_2', 'SJC_SAMPLE_2', 'IncFormLen', 'SkipFormLen']

all_files = [file for file in glob.glob("/scratch/jve4pt/**/*JunctionCountOnly.txt", recursive=True) if "MATS" in file ]
len(all_files)

all_files

4220

['/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/A5SS.MATS.JunctionCountOnly.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/MXE.MATS.JunctionCountOnly.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/SE.MATS.JunctionCountOnly.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/A3SS.MATS.JunctionCountOnly.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/RI.MATS.JunctionCountOnly.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/A3SS.MATS.JunctionCountOnly.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/RI.MATS.JunctionCountOnly.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/SE.MATS.JunctionCountOnly.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/MXE.MATS.JunctionCountOnly.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/A5SS.MATS.JunctionCountOnly.txt',
 '/scratch/jve4pt/CUGBP1-BGHLV23-HepG2/MATS_Norm_output/A5SS.MATS.JunctionCountOnly.txt',
 '/scratch/jve4pt/CUGBP1-BGHLV23-HepG2/MATS_Norm_output

In [3]:
all_files[-1]

'/scratch/jve4pt/METAP2-BGKLV36-K562/MATS_output/RI.MATS.JunctionCountOnly.txt'

## Subset Files to Necessary Columns and Output

In [ ]:
for file in all_files: 
    tmp_df = pd.read_csv(file, sep="\t")
        
    tmp_df[columns_to_keep].to_csv(
        file.replace(".txt", "_TMP_SUBSET.tsv"), 
        sep="\t", 
        index=False
    )
    
    

## SLURM and other parameters for script generation below

In [4]:
partition="standard" 
memory="6GB"
cpu="1"

script_path="/home/jve4pt/rMATS-STAT/rMATS_unpaired.py"
diff_cutoff=0.0001

## Create Submission Scripts for P-value Recalculation

In [5]:
subsetted_files = glob.glob("/scratch/jve4pt/**/*SUBSET.tsv", recursive=True)
subsetted_files

['/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/A3SS.MATS.JunctionCountOnly_TMP_SUBSET.tsv',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/A5SS.MATS.JunctionCountOnly_TMP_SUBSET.tsv',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/RI.MATS.JunctionCountOnly_TMP_SUBSET.tsv',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/MXE.MATS.JunctionCountOnly_TMP_SUBSET.tsv',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/SE.MATS.JunctionCountOnly_TMP_SUBSET.tsv',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/SE.MATS.JunctionCountOnly_TMP_SUBSET.tsv',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/A5SS.MATS.JunctionCountOnly_TMP_SUBSET.tsv',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/A3SS.MATS.JunctionCountOnly_TMP_SUBSET.tsv',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/RI.MATS.JunctionCountOnly_TMP_SUBSET.tsv',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/MXE.MATS.JunctionCountOnly_TMP_SUBSET.tsv',
 '/scratch/jve4pt/CUGBP1-BGHLV23-He

In [ ]:
for file in subsetted_files: 
    with open(file.replace(".tsv", "_SUBMISSION.sh"), 'w') as out_file: 
        _=out_file.write("#!/bin/bash\n")
        _=out_file.write(
            "#SBATCH --output={}\n".format(
                file.replace(".tsv", "_output.txt")
            )
        )
        _=out_file.write(
            "#SBATCH --error={}\n".format(
                file.replace(".tsv", "_error.txt")
            )
        )
        _=out_file.write("#SBATCH --partition={}\n".format(partition))
        _=out_file.write("#SBATCH --mem={}\n".format(memory))
        _=out_file.write("#SBATCH -n {}\n".format(cpu))
        _=out_file.write("#SBATCH --account=platiglab\n")

        _=out_file.write("module load gcc/11.4.0\n")
        _=out_file.write("module load python/2.7.18\n")
        
        _=out_file.write("python {} {} {}/ {} {}\n".format(
                
                script_path, 
                file,
                "/".join( file.split("/")[0:-1]), 
                cpu, 
                diff_cutoff
                
            )
        )
        
    
    os.system("sbatch {}".format(
            file.replace(".tsv", "_SUBMISSION.sh")
        )
    )


#### Re-submit Jobs that Timed Out 

In [ ]:
for file in subsetted_files: 
    error_file = file.replace(".tsv", "_error.txt")
    
    if os.path.getsize(error_file) > 0: 
        
        partition="parallel"
        memory="10GB"
        n_nodes = 2
        cpu=12
        script_path="/home/jve4pt/rMATS-STAT/rMATS_unpaired.py"
        diff_cutoff=0.0001
        
        with open(file.replace(".tsv", "_SUBMISSION.sh"), 'w') as out_file:
                
            _=out_file.write("#!/bin/bash\n")
            _=out_file.write(
                "#SBATCH --output={}\n".format(
                    file.replace(".tsv", "_output.txt")
                )
            )
            _=out_file.write(
                "#SBATCH --error={}\n".format(
                    file.replace(".tsv", "_error.txt")
                )
            )
            _=out_file.write("#SBATCH --partition={}\n".format(partition))
            _=out_file.write("#SBATCH --mem={}\n".format(memory))
            _=out_file.write("#SBATCH -N {}\n".format(n_nodes))
            _=out_file.write("#SBATCH -n {}\n".format(cpu))
            _=out_file.write("#SBATCH --account=platiglab\n")

            _=out_file.write("module load gcc/11.4.0\n")
            _=out_file.write("module load python/2.7.18\n")

            _=out_file.write("python {} {} {}/ {} {}\n".format(

                    script_path, 
                    file,
                    "/".join( file.split("/")[0:-1]), 
                    cpu, 
                    diff_cutoff

                )
            )

        os.system("sbatch {}".format(
                file.replace(".tsv", "_SUBMISSION.sh")
            )
        )

        

## Validate that Jobs Ran Properly

There should be no output from the below. 

In [6]:
! find /scratch/jve4pt/ -type f -iname "*output*" -exec cat '{}' \; 

In [7]:
! find /scratch/jve4pt/ -type f -iname "*error*" -exec cat '{}' \;

Check that there is a corrected p value file for every original results file 

In [8]:
for file in all_files: 
    if not os.path.exists(
        file.replace(
            ".txt", "rMATS_Result_P.txt"
        )
    ): 
        print(file)

## Do FDR Calculations

In [9]:
recalculated_files = glob.glob("/scratch/jve4pt/**/*Result_P*", recursive=True)
recalculated_files

['/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/A3SS.MATS.JunctionCountOnlyrMATS_Result_P.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/A5SS.MATS.JunctionCountOnlyrMATS_Result_P.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/SE.MATS.JunctionCountOnlyrMATS_Result_P.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/RI.MATS.JunctionCountOnlyrMATS_Result_P.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_output/MXE.MATS.JunctionCountOnlyrMATS_Result_P.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/RI.MATS.JunctionCountOnlyrMATS_Result_P.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/MXE.MATS.JunctionCountOnlyrMATS_Result_P.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/A5SS.MATS.JunctionCountOnlyrMATS_Result_P.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/SE.MATS.JunctionCountOnlyrMATS_Result_P.txt',
 '/scratch/jve4pt/BCCIP-BGHLV17-HepG2/MATS_Norm_output/A3SS.MATS.JunctionCountOnlyrMATS_Result_P.txt',
 '/sc

In [ ]:
# for each recalculated p value file
for recalculated_file in recalculated_files: 
    
    # take the recalculated and the original file 
    # set the index to event ID (for future joining)
    recalculated_df = pd.read_csv(recalculated_file, sep="\t").set_index("ID")
    incorrect_old_df = pd.read_csv(recalculated_file.replace("rMATS_Result_P.txt", ".txt"), sep="\t").set_index("ID")
    
    # save the original column order for use later
    original_column_order = incorrect_old_df.columns
    
    # calculate FDR with Benjamini-Hochberg method
    recalculated_df["FDR"] = scipy.stats.false_discovery_control(
        recalculated_df["PValue"].to_list(), 
        method="bh"
    )
    
    # get headers in original file not found in recalculated pval file 
    # subset to those columns
    subset_columns = [header for header in original_column_order if header not in recalculated_df.columns]
    incorrect_old_df = incorrect_old_df[subset_columns]

    # left join new p values and FDR values to the original table
    final_output_df = incorrect_old_df.join(recalculated_df, how="left")
    
    # order the columns back to the original way it's supposed to be 
    final_output_df = final_output_df[original_column_order]
    
    # output tsv files 
    final_output_df.to_csv(
        recalculated_file.replace(
            "rMATS_Result_P.txt", 
            "_corrected_pval_yogi_february_2024.tsv"
        ), 
        
        sep="\t", 
    )

## Check that there were no problems with joining

In [20]:
for file in glob.glob("/scratch/jve4pt/**/*yogi*", recursive=True): 
    
    tmp_df = pd.read_csv(file, sep="\t")
    
    # check if any value is "N/A"
    # then checks if any value per column has an "N/A"
    # then checks if any column has an "N/A"
    if tmp_df.isna().any().any(): 
        file

## Remove all intermediary files 

In [9]:
intermediary_file_keywords = ["TMP_SUBSET", "Result_P.txt"]

for keyword in intermediary_file_keywords: 
    for file in glob.glob("/scratch/jve4pt/**/*{}*".format(keyword), recursive=True):
        os.remove(file)